# MARSNet — LSTM Baseline

## What this notebook tests
Replacing the **TCN + ALiBi attention** temporal backbone with a **2-layer unidirectional LSTM**.
Everything else is identical to R20 (paper model): WindowEncoder, OutageStepPE, VelocityHead,
loss function (v10 gyro-gated L_cvprior, LAM_CVPRIOR=0.50), training protocol.

## What changes vs R20
| Component | R20 (MARSNet) | LSTM Baseline |
|-----------|--------------|---------------|
| Temporal backbone | TCNBackbone (dilations 1,2,4,8,16 × 2 stacks) + ALiBiCausalAttention | LSTMBackbone (2-layer, hidden=128) |
| Everything else | — | **IDENTICAL** |

## Expected outcomes
- **LSTM ≈ MARSNet**: L_cvprior drives performance, backbone is secondary.
  → Paper frames physics-constrained loss as the main contribution.
- **LSTM << MARSNet**: TCN+ALiBi provides additional architectural benefit.
  → Strengthens the architecture claim. Long-outage group (S53–S58) is the diagnostic.

## R20 reference numbers (to beat or match)
```
Mean=6.56m  Median=0.48m  90th=32.52m  79.7%<5m
S0-34=0.38m  S35-40=2.28m  S41-46=2.97m  S47-52=32.56m  S53-58=24.52m
TCN ablation=92.36x
```

In [ ]:
import math, os, time, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import WeightedRandomSampler, DataLoader
from collections import defaultdict
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Configuration (portable — no Colab / Google Drive required) ───────────────
# Point GATEIO_DATA at MARS_Master_Dataset.npz, or edit the default path below.
import os
DATA_PATH = os.environ.get("GATEIO_DATA", "../data/processed/MARS_Master_Dataset.npz")
CKPT_DIR  = os.environ.get("GATEIO_CKPT", "./checkpoints")
PLOT_DIR  = os.environ.get("GATEIO_OUT",  "./results")
CKPT_PATH = os.path.join(CKPT_DIR, "marsnet_lstm_best.pt")
CKPT_LAST = os.path.join(CKPT_DIR, "marsnet_lstm_last.pt")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Architecture ───────────────────────────────────────────────────────────────
D_MODEL    = 48
N_HEADS    = 4
DROPOUT    = 0.15
N_CHAN     = 14
N_IMU_CHAN = 10
SEQ_LEN    = 300
WIN_LEN    = 200
DT         = 0.1

# ── LSTM-specific ──────────────────────────────────────────────────────────────
LSTM_HIDDEN  = 128   # hidden size per direction
LSTM_LAYERS  = 2     # number of stacked LSTM layers
LSTM_DROPOUT = 0.15  # dropout between LSTM layers (not applied after final layer)

# ── Loss weights — IDENTICAL to R20 ───────────────────────────────────────────
LAM_DR          = 0.90
LAM_CVPRIOR     = 0.50   # R20 value — do NOT change for a fair baseline comparison
CVPRIOR_GYRO_THR= 0.10   # rad/s — from R19 empirical IMU distribution
LAM_VAR         = 0.0
LAM_SMOOTH      = 0.001
LAM_DRIFT       = 0.002
DR_AXIS_WEIGHTS = torch.tensor([1.0, 1.0, 3.0])
HUBER_DELTA     = 0.3

# ── Warmup ─────────────────────────────────────────────────────────────────────
PHYS_WARM_START, PHYS_WARM_END   = 30, 60
DRIFT_WARM_START, DRIFT_WARM_END = 30, 70
LAM_PHYS_MAX = 0.01

# ── Thresholds ─────────────────────────────────────────────────────────────────
TURN_GYRO_THR = 0.10
ZUPT_GYRO_THR = 0.05

# ── Augmentation — IDENTICAL to R20 ───────────────────────────────────────────
TEMPORAL_DROPOUT_P = 0.25
W_TURN             = 8.0
P_VPREV_MASK       = 0.0   # never use

# ── Training — IDENTICAL to R20 ───────────────────────────────────────────────
MAX_EPOCHS  = 200
PATIENCE    = 60
LR          = 8e-4
BATCH_SIZE  = 16

print('Config loaded — LSTM Baseline')
print(f'  LSTM: hidden={LSTM_HIDDEN}, layers={LSTM_LAYERS}, dropout={LSTM_DROPOUT}')
print(f'  Loss: LAM_CVPRIOR={LAM_CVPRIOR} (R20 value, gyro-only gate, no truth gate)')
print(f'  Everything else: IDENTICAL to R20')

In [ ]:
# ── Shared components: identical to MARSNet R20/R22 ───────────────────────────

class OutageStepPE(nn.Module):
    def __init__(self, d_model, max_steps=211):
        super().__init__()
        pe = torch.zeros(max_steps, d_model)
        pos = torch.arange(max_steps).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
    def forward(self, tokens, outage_flag):
        flag = (outage_flag > 0.5).long()
        cumsum = flag.cumsum(dim=1)
        reset = cumsum * (1 - flag)
        steps = (cumsum - reset.cummax(dim=1).values).clamp(0, self.pe.shape[0] - 1)
        return tokens + self.pe[steps]

class WindowEncoder(nn.Module):
    def __init__(self, in_channels=N_CHAN, d_model=D_MODEL):
        super().__init__()
        g = min(8, d_model // 6)
        self.conv1 = nn.Conv1d(in_channels, d_model, 7, padding=3); self.norm1 = nn.GroupNorm(g, d_model)
        self.conv2 = nn.Conv1d(d_model, d_model, 5, padding=2);     self.norm2 = nn.GroupNorm(g, d_model)
        self.conv3 = nn.Conv1d(d_model, d_model, 3, padding=1);     self.norm3 = nn.GroupNorm(g, d_model)
        self.drop  = nn.Dropout(0.1)
        self.cls   = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pool  = nn.MultiheadAttention(d_model, num_heads=4, dropout=0.1, batch_first=True)
    def forward(self, x):
        x = F.gelu(self.norm1(self.conv1(x)))
        x = F.gelu(self.norm2(self.conv2(x)))
        x = F.gelu(self.norm3(self.conv3(x)))
        x = self.drop(x.transpose(1, 2))
        out, _ = self.pool(self.cls.expand(x.size(0), -1, -1), x, x)
        return out.squeeze(1)

class VelocityHead(nn.Module):
    def __init__(self, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        h = d_model // 2
        def _b(): return nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, h),
                                        nn.GELU(), nn.Dropout(dropout), nn.Linear(h, 3))
        self.head_aided  = _b()
        self.head_dr     = _b()
        self.v_prev_proj = nn.Linear(3, d_model)
    def forward(self, tokens, outage_flag, v_prev):
        alpha = outage_flag.unsqueeze(-1)
        return (1. - alpha) * self.head_aided(tokens) + alpha * self.head_dr(tokens + alpha * self.v_prev_proj(v_prev))

# ── LSTM backbone: THE ONLY CHANGE vs MARSNet ─────────────────────────────────

class LSTMBackbone(nn.Module):
    """
    Unidirectional 2-layer LSTM replacing TCNBackbone + ALiBiCausalAttention.
    Output is projected back to D_MODEL so VelocityHead is unchanged.
    Causal by construction: LSTM only sees past context at each step.
    """
    def __init__(self, d_model=D_MODEL, hidden=LSTM_HIDDEN, n_layers=LSTM_LAYERS, dropout=LSTM_DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = d_model,
            hidden_size = hidden,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0
            # bidirectional=False is the default — must stay False (causal requirement)
        )
        self.proj = nn.Linear(hidden, d_model)

    def forward(self, x, src_key_padding_mask=None):
        # x: (B, S, D_MODEL)
        # src_key_padding_mask: accepted for API compatibility but not used (LSTM is inherently causal)
        out, _ = self.lstm(x)   # (B, S, hidden)
        return self.proj(out)   # (B, S, D_MODEL)

# ── LSTM-MARSNet: same forward() signature as MARSNet ─────────────────────────

class MARSNetLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.window_enc = WindowEncoder(N_CHAN, D_MODEL)  # identical
        self.pos_enc    = OutageStepPE(D_MODEL)           # identical
        self.backbone   = LSTMBackbone()                  # ← only change
        self.head       = VelocityHead(D_MODEL, DROPOUT)  # identical

    def forward(self, x, outage_flag, v_prev):
        B, S, W, C = x.shape
        tokens = self.window_enc(x.reshape(B * S, W, C).permute(0, 2, 1).contiguous()).view(B, S, -1)
        tokens = self.pos_enc(tokens, outage_flag)
        tokens = self.backbone(tokens)
        return self.head(tokens, outage_flag, v_prev)

print('MARSNetLSTM defined.')
print('  WindowEncoder + OutageStepPE + VelocityHead: IDENTICAL to R20')
print('  Backbone: LSTMBackbone (2-layer LSTM, hidden=128) replaces TCN+ALiBi')

In [ ]:
# ── Loss v10: gyro-gated L_cvprior — IDENTICAL to R20 ─────────────────────────
# truth_gate_active is always False here (R20 mode, no GPS-constant gate)

def combined_loss_v10_batched(dv_pred_norm, dv_true_norm,
                               outage_mask, dv_iqr_t, dv_median_t,
                               v_prev=None, x_raw=None,
                               lam_p=0.0, lam_d=0.0, cap_m=100.0):
    """
    R20 loss (v10): gyro-gated constant-velocity prior.
    Gate: |gyro_z| < CVPRIOR_GYRO_THR (0.10 rad/s) AND window is outage.
    No truth gate (that was R22 — it made things worse).
    LAM_CVPRIOR=0.50 < LAM_DR=0.90 (R20 optimal balance).
    """
    B, S, _ = dv_pred_norm.shape
    device = dv_pred_norm.device
    iqr = dv_iqr_t.to(device); med = dv_median_t.to(device)
    aw  = DR_AXIS_WEIGHTS.to(device)
    out_mask = outage_mask > 0.5
    aid_mask = ~out_mask

    L_data = (F.huber_loss(dv_pred_norm[aid_mask][:, :2], dv_true_norm[aid_mask][:, :2],
                            delta=HUBER_DELTA, reduction='mean')
               if aid_mask.any() else torch.tensor(0., device=device))

    L_dr = (F.huber_loss(dv_pred_norm[out_mask] * aw, dv_true_norm[out_mask] * aw,
                          delta=HUBER_DELTA, reduction='mean')
             if out_mask.any() else torch.tensor(0., device=device))

    # L_cvprior: gyro-only gate (R20 mode)
    L_cvprior = torch.tensor(0., device=device); n_straight = 0
    if x_raw is not None and out_mask.any() and v_prev is not None:
        gyro_z      = x_raw[:, :, WIN_LEN // 2, 5].abs()
        straight_out = (gyro_z < CVPRIOR_GYRO_THR) & out_mask
        n_straight   = straight_out.sum().item()
        if straight_out.any():
            vp_norm   = (v_prev.to(device) - med) / (iqr + 1e-8)
            L_cvprior = F.huber_loss(
                dv_pred_norm[straight_out][:, :2],
                vp_norm[straight_out][:, :2].detach(),
                delta=HUBER_DELTA, reduction='mean'
            )

    L_smooth = (dv_pred_norm[:, 1:, :] - dv_pred_norm[:, :-1, :]).pow(2).mean()

    L_phys = torch.tensor(0., device=device)
    if lam_p > 0 and x_raw is not None:
        gm = x_raw[:, :, WIN_LEN // 2, 3:6].norm(dim=-1)
        zm = (gm < ZUPT_GYRO_THR) & out_mask
        if zm.any(): L_phys = (dv_pred_norm[zm].norm(dim=-1) * gm[zm]).mean()

    tl = [dv_pred_norm[b, out_mask[b].nonzero(as_tuple=True)[0][0]].pow(2).mean()
          for b in range(B) if out_mask[b].any()]
    L_trans = torch.stack(tl).mean() if tl else torch.tensor(0., device=device)

    L_drift = torch.tensor(0., device=device)
    if lam_d > 0 and out_mask.any():
        dl = []
        for b in range(B):
            oi = out_mask[b].nonzero(as_tuple=True)[0]
            if len(oi) == 0: continue
            pm = dv_pred_norm[b, oi] * iqr + med
            tm = dv_true_norm[b, oi] * iqr + med
            pe = ((pm - tm) * DT).cumsum(0).norm(dim=-1)
            dl.append(pe[-1].clamp(max=cap_m))
        if dl: L_drift = torch.stack(dl).mean()

    total = (L_data + LAM_DR * L_dr + LAM_CVPRIOR * L_cvprior
             + LAM_SMOOTH * L_smooth + lam_p * L_phys
             + LAM_DRIFT * lam_d * L_drift + 0.05 * L_trans)

    return total, {'L_data': L_data.item(), 'L_dr': L_dr.item(),
                   'L_cvprior': L_cvprior.item(), 'L_phys': L_phys.item(),
                   'L_smooth': L_smooth.item(), 'L_drift': L_drift.item(),
                   'L_trans': L_trans.item(), 'n_straight': float(n_straight)}

print('Loss v10 (R20 gyro-gated L_cvprior) defined.')
print(f'  LAM_CVPRIOR={LAM_CVPRIOR}  LAM_DR={LAM_DR}  CVPRIOR_GYRO_THR={CVPRIOR_GYRO_THR} rad/s')
print('  No truth gate. Identical to R20.')

In [ ]:
import sys, os as _os
sys.path.insert(0, _os.path.abspath(_os.path.join('..', 'data')))
from data_loader import MARSDataset

_npz = np.load(DATA_PATH)
dv_iqr    = _npz['Y_iqr'].astype(np.float32)
dv_median = _npz['Y_median'].astype(np.float32)
dv_iqr_t  = torch.from_numpy(dv_iqr).to(DEVICE)
dv_med_t  = torch.from_numpy(dv_median).to(DEVICE)
DV_IQR_TRUE = np.array([0.055, 0.319, 0.00078], dtype=np.float32)
print(f'Y_iqr={dv_iqr}  Y_median={dv_median}')
print(f'NOTE: model predicts GPS velocity (Y_iqr space). v_pred=0 → v_y={dv_median[1]:.3f} m/s physical!')

train_ds = MARSDataset(DATA_PATH, split='train', outage_prob=0.8)
val_ds   = MARSDataset(DATA_PATH, split='val',   outage_prob=0.0)
X_tr = _npz['X_train']; W_train_flat = _npz['W_train']; train_valid_idx = _npz['train_valid_idx']
N_SEQS = len(train_valid_idx)

sample_weights = np.zeros(N_SEQS, dtype=np.float32)
for i, start in enumerate(train_valid_idx):
    end   = min(int(start) + SEQ_LEN, len(X_tr))
    gyro  = np.linalg.norm(X_tr[int(start):end, WIN_LEN // 2, 3:6], axis=-1)
    sample_weights[i] = (W_TURN if np.any(gyro > TURN_GYRO_THR)
                         else float(np.mean(W_train_flat[int(start):end])))
sample_weights /= sample_weights.sum()

sampler      = WeightedRandomSampler(torch.from_numpy(sample_weights).float(),
                                      num_samples=N_SEQS, replacement=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
n_batches    = math.ceil(N_SEQS / BATCH_SIZE)
print(f'Train:{N_SEQS}  Val:{len(val_ds)}  Batches:{n_batches}')

In [ ]:
def compute_v_prev(y_norm, outage_mask, dv_iqr_t, dv_med_t):
    device = y_norm.device; B, S, _ = y_norm.shape
    v_raw    = y_norm * dv_iqr_t + dv_med_t
    not_flag = ~(outage_mask > 0.5)
    t_idx    = torch.arange(S, device=device).unsqueeze(0).expand(B, -1)
    gps_idx  = torch.where(not_flag, t_idx, torch.zeros_like(t_idx))
    last_gps = gps_idx.cummax(dim=1).values
    return torch.stack([v_raw[:, :, ax].gather(1, last_gps) for ax in range(3)], dim=-1), v_raw
print('compute_v_prev defined.')

In [ ]:
torch.manual_seed(42)
model   = MARSNetLSTM().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MARSNetLSTM parameters: {n_params:,}')
print(f'MARSNet R20 parameters:  186,390')
print(f'Difference: {n_params - 186390:+,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=MAX_EPOCHS * n_batches,
    pct_start=0.05, anneal_strategy='cos', div_factor=25, final_div_factor=100)

# Speed benchmark
model.eval()
_b   = next(iter(train_loader))
_xn  = _b['x_norm'].to(DEVICE); _om = _b['outage_mask'].to(DEVICE); _yn = _b['y_norm'].to(DEVICE)
B, S, W, _ = _xn.shape
_vp  = torch.zeros(B, S, 3, device=DEVICE)
_flag = _om.float()
_vpch = _vp.unsqueeze(2).expand(-1, -1, W, -1)
_fch  = _flag.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, W, 1)
_xf   = torch.cat([_xn[:, :, :, :N_IMU_CHAN], _vpch, _fch], dim=-1)
with torch.no_grad():
    for _ in range(3): model(_xf, _flag, _vp)
if DEVICE.type == 'cuda': torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    for _ in range(10): model(_xf, _flag, _vp)
if DEVICE.type == 'cuda': torch.cuda.synchronize()
ms = (time.time() - t0) * 100
print(f'Forward:{ms:.1f}ms  Est epoch:{ms / 1000 * n_batches * 2.2:.0f}s')

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, dv_iqr_t, dv_median_t, device, lam_p=0., lam_d=0.):
    model.train(); totals = defaultdict(float); n = 0
    for batch in loader:
        xn = batch['x_norm'].to(device); yn = batch['y_norm'].to(device)
        om = batch['outage_mask'].to(device); xr = batch['x_raw'].to(device)
        B, S, W, _ = xn.shape
        vp, _ = compute_v_prev(yn, om, dv_iqr_t.to(device), dv_median_t.to(device))
        flag = om.float(); xi = xn[:, :, :, :N_IMU_CHAN]
        if TEMPORAL_DROPOUT_P > 0:
            xi = xi * (torch.rand(B, S, 1, 1, device=device) > TEMPORAL_DROPOUT_P).float()
        vpch = vp.unsqueeze(2).expand(-1, -1, W, -1)
        fch  = flag.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, W, 1)
        xf   = torch.cat([xi, vpch, fch], dim=-1)
        optimizer.zero_grad()
        dv_pred = model(xf, flag, vp)
        loss, comps = combined_loss_v10_batched(
            dv_pred, yn, om, dv_iqr_t, dv_median_t,
            v_prev=vp, x_raw=xr, lam_p=lam_p, lam_d=lam_d)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        totals['total'] += loss.item()
        for k, v in comps.items(): totals[k] += v
        n += 1
    return {k: v / max(n, 1) for k, v in totals.items()}
print('train_one_epoch defined.')

In [ ]:
history = {k: [] for k in ['train_loss', 'val_drift_m', 'turn_rmse', 'straight_rmse',
    'L_data', 'L_dr', 'L_cvprior', 'L_smooth', 'L_drift',
    'sr_x', 'sr_y', 'L_cvprior_val', 'mean_vprev_err_ms', 'gate_pct_val']}
best_drift = float('inf'); best_state = None; patience_ctr = 0; start_epoch = 1

print(f"{'Ep':>4} {'Loss':>8} {'Drift':>8} {'L_cvp':>7} {'VpErr':>7} {'Gate%':>7} {'sr_y':>6} {'t(s)':>6}  Status")
print('-' * 82)

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    ep_t0 = time.time()
    lam_p = min(LAM_PHYS_MAX, LAM_PHYS_MAX * max(0, epoch - PHYS_WARM_START) / max(1, PHYS_WARM_END - PHYS_WARM_START))
    lam_d = min(1., max(0., epoch - DRIFT_WARM_START) / max(1, DRIFT_WARM_END - DRIFT_WARM_START))
    tc = train_one_epoch(model, train_loader, optimizer, scheduler, dv_iqr_t, dv_med_t, DEVICE, lam_p, lam_d)

    model.eval()
    val_drifts, sr_xs, sr_ys, turn_e, str_e, vprev_vals, vprev_err, gp_vals = [], [], [], [], [], [], [], []
    with torch.no_grad():
        for batch in val_loader:
            xn = batch['x_norm'].to(DEVICE); yn = batch['y_norm'].to(DEVICE)
            om = batch['outage_mask'].to(DEVICE); xr = batch['x_raw'].to(DEVICE)
            B, S, W, _ = xn.shape
            vp, _ = compute_v_prev(yn, om, dv_iqr_t, dv_med_t)
            flag = om.float(); xi = xn[:, :, :, :N_IMU_CHAN]
            vpch = vp.unsqueeze(2).expand(-1, -1, W, -1)
            fch  = flag.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, W, 1)
            xf   = torch.cat([xi, vpch, fch], dim=-1)
            dvp  = model(xf, flag, vp)
            omm  = om > 0.5
            gz = xr[:, :, WIN_LEN // 2, 5].abs(); str_v = (gz < CVPRIOR_GYRO_THR) & omm
            if omm.any(): gp_vals.append(100 * str_v.sum().item() / omm.sum().item())
            if str_v.any():
                vp_n = (vp - dv_med_t) / (dv_iqr_t + 1e-8)
                lv = F.huber_loss(dvp[str_v][:, :2], vp_n[str_v][:, :2].detach(),
                                   delta=HUBER_DELTA, reduction='mean').item()
                vprev_vals.append(lv)
                vprev_err.append((dvp[str_v][:, 1] - vp_n[str_v][:, 1]).abs().mean().item() * dv_iqr_t[1].item())
            for b in range(B):
                oi = om[b].nonzero(as_tuple=True)[0]
                if len(oi) == 0: continue
                pm = dvp[b] * dv_iqr_t + dv_med_t; tm = yn[b] * dv_iqr_t + dv_med_t
                pe = ((pm - tm) * DT)[oi].cumsum(0).norm(dim=-1)
                val_drifts.append(pe[-1].item())
                ps = dvp[b, oi].std(0); ts = yn[b, oi].std(0)
                sr_xs.append((ps[0] / (ts[0] + 1e-8)).item())
                sr_ys.append((ps[1] / (ts[1] + 1e-8)).item())
                gm = xr[b, :, W // 2, 3:6].norm(dim=-1)
                for idx in oi:
                    e2 = (pm[idx] - tm[idx]).pow(2).mean().item()
                    if gm[idx] > TURN_GYRO_THR: turn_e.append(e2)
                    elif gm[idx] >= ZUPT_GYRO_THR: str_e.append(e2)

    vd    = float(np.mean(val_drifts)) if val_drifts else float('nan')
    sry   = float(np.mean(sr_ys)) if sr_ys else float('nan')
    lv_v  = float(np.mean(vprev_vals)) if vprev_vals else float('nan')
    vp_ms = float(np.mean(vprev_err)) if vprev_err else float('nan')
    gp    = float(np.mean(gp_vals)) if gp_vals else float('nan')
    ep_s  = time.time() - ep_t0

    history['train_loss'].append(tc['total']); history['val_drift_m'].append(vd)
    history['sr_y'].append(sry); history['L_cvprior_val'].append(lv_v)
    history['mean_vprev_err_ms'].append(vp_ms); history['gate_pct_val'].append(gp)
    for k in ['L_data', 'L_dr', 'L_cvprior', 'L_smooth', 'L_drift']:
        history[k].append(tc.get(k, 0.))

    is_best = vd < best_drift
    if is_best:
        best_drift = vd; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
        torch.save({'model': best_state, 'epoch': epoch, 'best_val': best_drift,
                    'DV_iqr': dv_iqr, 'DV_median': dv_median, 'history': history, 'run': 'lstm'}, CKPT_PATH)
    else:
        patience_ctr += 1

    if epoch == 1 or epoch % 5 == 0 or is_best:
        status = f'★ BEST {vd:.2f}m' if is_best else f'({patience_ctr}/{PATIENCE})'
        print(f'{epoch:>4}  {tc["total"]:>8.4f}  {vd:>7.2f}m  {tc.get("L_cvprior",0):>7.4f}  {vp_ms:>7.3f}  {gp:>6.1f}%  {sry:>6.2f}  {ep_s:>6.0f}  {status}')

    if epoch == 30:
        print(f'  ⚠ Ep30 VpErr={vp_ms:.3f} (R20 was 0.42 at ep30)')
        print(f'  ⚠ Ep30 drift={vd:.1f}m (R20 was 14.8m at ep20)')

    if patience_ctr >= PATIENCE:
        print(f'Early stop epoch {epoch}')
        break

print(f'\nBest: {best_drift:.3f}m  (R20: 6.56m  Naive: 5.96m)')

In [ ]:
def save_training_history(history, save_path=None):
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].plot(history.get('train_loss', [])); axes[0].set_title('Train Loss')
    axes[1].plot(history.get('val_drift_m', []))
    axes[1].axhline(5.0, color='g', ls='--', lw=1)
    axes[1].axhline(6.56, color='orange', ls=':', lw=1, label='R20 best')
    axes[1].legend(fontsize=8); axes[1].set_title('Val Drift (m)')
    axes[2].plot(history.get('L_cvprior', []), label='train')
    axes[2].plot(history.get('L_cvprior_val', []), label='val', ls='--')
    axes[2].axhline(0.01, color='g', ls='--', lw=1)
    axes[2].set_title('L_cvprior'); axes[2].legend(fontsize=8)
    axes[3].plot(history.get('mean_vprev_err_ms', []))
    axes[3].axhline(0.05, color='g', ls='--', lw=1, label='target')
    axes[3].axhline(0.258, color='r', ls=':', lw=1, label='R20')
    axes[3].set_title('VpErr |v_pred_y−v_prev_y| m/s'); axes[3].legend(fontsize=8)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show(); plt.close()

save_training_history(history, os.path.join(PLOT_DIR, 'training_history.png'))
print(f'Plots → {PLOT_DIR}/')

In [ ]:
@torch.no_grad()
def predict_sequence(model, x_norm_seq, y_raw_seq, outage_start, outage_end, dv_iqr, dv_median, device):
    model.eval(); S = x_norm_seq.shape[0]; W = x_norm_seq.shape[1]
    om = torch.zeros(S, dtype=torch.float32)
    if outage_start >= 3: om[outage_start - 2] = 1/3; om[outage_start - 1] = 2/3
    om[outage_start:outage_end] = 1.
    vr = torch.from_numpy(y_raw_seq).float(); vp = torch.zeros(S, 3); last = vr[0].clone()
    for t in range(S):
        if om[t] > 0.5: vp[t] = last
        else: vp[t] = vr[t]; last = vr[t].clone()
    xi   = torch.from_numpy(x_norm_seq).float()[:, :, :N_IMU_CHAN]
    vpch = vp.unsqueeze(1).expand(-1, W, -1)
    fch  = om.view(S, 1, 1).expand(-1, W, 1)
    xf   = torch.cat([xi, vpch, fch], dim=-1)
    dvp  = model(xf.unsqueeze(0).to(device), om.unsqueeze(0).to(device), vp.unsqueeze(0).to(device)).squeeze(0).cpu()
    iqr  = torch.from_numpy(dv_iqr).float(); med = torch.from_numpy(dv_median).float()
    pm   = dvp * iqr + med; tm = vr
    def integrate(dv, s):
        pos = torch.zeros(S, 2)
        for t in range(s + 1, S): pos[t] = pos[t-1] + dv[t, :2] * DT
        return pos.numpy()
    pp = integrate(pm, outage_start); pt = integrate(tm, outage_start)
    return {'dv_pred_ms': pm.numpy(), 'dv_true_ms': tm.numpy(), 'pos_pred': pp, 'pos_true': pt,
            'outage_start': outage_start,
            'drift_m': float(np.linalg.norm(pp[outage_end-1] - pt[outage_end-1]))}

def plot_grid(eval_results, save_path, n_cols=6):
    n = len(eval_results); nr = math.ceil(n / n_cols)
    fig, axes = plt.subplots(nr, n_cols, figsize=(n_cols * 3.2, nr * 2.8), squeeze=False)
    for i, res in enumerate(eval_results):
        ax = axes[i // n_cols][i % n_cols]
        gt = res['pos_gt']; pred = res['pos_pred']; oi = res['outage_start']
        ax.plot(gt[:oi, 1], gt[:oi, 0], 'b-', lw=1.)
        ax.plot(gt[oi:, 1], gt[oi:, 0], 'b--', lw=1., alpha=0.5)
        ax.plot(pred[oi:, 1], pred[oi:, 0], 'r-', lw=1.)
        ax.plot(gt[oi, 1], gt[oi, 0], 'ko', ms=3, zorder=5)
        d = res['drift_m']; c = 'green' if d < 5 else ('darkorange' if d < 15 else 'red')
        beats = res.get('naive_drift_m', 999) > d
        ax.set_title(f"S{res['seq_idx']} {'✓' if beats else '✗'} {d:.1f}m", fontsize=7.5, color=c, fontweight='bold')
        ax.set_aspect('equal'); ax.tick_params(labelsize=5); ax.grid(True, alpha=0.25)
    for j in range(n, nr * n_cols): axes[j // n_cols][j % n_cols].set_visible(False)
    fig.suptitle('LSTM Baseline | Green<5m  Orange<15m  Red≥15m', fontsize=10)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
print('predict_sequence + grid defined.')

In [ ]:
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

_nv = np.load(DATA_PATH); Xv = _nv['X_val']; Yv = _nv['Y_val']
Xmed = _nv['X_median']; Xiq = _nv['X_iqr']; vi = _nv['val_valid_idx']
SL = int(_nv['seq_len'][0]) if 'seq_len' in _nv else SEQ_LEN
GROUP = {**{i: 'straight-short' for i in range(0, 35)},
         **{i: 'straight-med'   for i in range(35, 41)},
         **{i: 'TURN'           for i in range(41, 47)},
         **{i: 'FALSE-ALARM'    for i in range(47, 53)},
         **{i: 'long-outage'    for i in range(53, 59)}}

all_res = []; mdrifts = []; ndrifts = []
print(f"{'S':>3}  {'Model':>7}  {'Naive':>7}  Group"); print('-' * 40)
for si, start in enumerate(vi):
    xrs = Xv[int(start):int(start)+SL]; yrs = Yv[int(start):int(start)+SL]
    if len(xrs) < SL: continue
    xns = (xrs - Xmed) / np.where(Xiq < 1e-6, 1., Xiq)
    os_ = SL // 3; oe_ = min(os_ + 100, SL)
    r   = predict_sequence(model, xns, yrs, os_, oe_, dv_iqr, dv_median, DEVICE)
    lv  = yrs[os_ - 1] if os_ > 0 else np.zeros(3)
    pn  = np.zeros((SL, 2))
    for t in range(os_ + 1, SL): pn[t] = pn[t-1] + lv[:2] * DT
    nd  = float(np.linalg.norm(pn[oe_ - 1] - r['pos_true'][oe_ - 1]))
    mdrifts.append(r['drift_m']); ndrifts.append(nd)
    all_res.append({'seq_idx': si,
                    'pos_gt':   np.column_stack([r['pos_true'], np.zeros(SL)]),
                    'pos_pred': np.column_stack([r['pos_pred'], np.zeros(SL)]),
                    'drift_m': r['drift_m'], 'naive_drift_m': nd,
                    'outage_start': os_, 'group': GROUP.get(si, '?')})
    print(f'{si:>3}  {r["drift_m"]:>6.2f}m  {nd:>6.2f}m  {GROUP.get(si, "?")}')

ma = np.array(mdrifts); na = np.array(ndrifts)
print('\n' + '=' * 58)
print(f'  Mean:{ma.mean():.2f}m  Median:{np.median(ma):.2f}m  90th:{np.percentile(ma, 90):.2f}m')
print(f'  % beats naive:{100*np.mean(ma<na):.1f}%  % under 5m:{100*np.mean(ma<5):.1f}%')
print(f'  R20: 6.56m  0.48m  32.52m  79.7% under 5m  [paper model]')
print(f'\n  ── KEY GROUPS ──')
for grp in ['straight-short', 'straight-med', 'TURN', 'FALSE-ALARM', 'long-outage']:
    idx = [i for i, r in enumerate(all_res) if r['group'] == grp]
    if not idx: continue
    gm = np.array([mdrifts[i] for i in idx]); gn = np.array([ndrifts[i] for i in idx])
    print(f'  {grp:<16}: {gm.mean():.2f}m  (naive={gn.mean():.2f}m  beats={np.mean(gm<gn)*100:.0f}%)')
r20_t = [4.44, 4.17, 3.33, 2.32, 1.70, 1.86]
r20_fa = [32.58, 32.50, 32.54, 32.58, 32.52, 32.65]
print(f'\n  ── TURNS S41-46 (R20=2.97m) ──')
for i, r in enumerate([r for r in all_res if r['group'] == 'TURN']):
    v = '✓' if r['drift_m'] < 5 else ('~' if r['drift_m'] < 10 else '✗')
    print(f'  S{r["seq_idx"]}: {r["drift_m"]:.2f}m  (R20={r20_t[i]:.2f}m)  {v}')
print(f'\n  ── FALSE-ALARM S47-52 (R20=32.5m) ──')
for i, r in enumerate([r for r in all_res if r['group'] == 'FALSE-ALARM']):
    v = '✓ FIXED' if r['drift_m'] < 5 else ('~ improved' if r['drift_m'] < 20 else '✗ still bad')
    print(f'  S{r["seq_idx"]}: {r["drift_m"]:.2f}m  (R20={r20_fa[i]:.1f}m)  {v}')
print('=' * 58)

plot_grid(all_res, os.path.join(PLOT_DIR, 'path_grid_all_seqs.png'), n_cols=6)

In [ ]:
print('=' * 65); print('  LSTM BASELINE DIAGNOSTICS'); print('=' * 65)
_nv2 = np.load(DATA_PATH); Xv2 = _nv2['X_val']; Yv2 = _nv2['Y_val']; vi2 = _nv2['val_valid_idx']
norm_d = []; norm_preds = []; norm_trues = []; norm_vprevs = []
model.eval()
with torch.no_grad():
    for res in all_res[:10]:
        si = res['seq_idx']; st = int(vi2[si])
        xrs = Xv2[st:st+SL]; yrs = Yv2[st:st+SL]
        xns = (xrs - Xmed) / np.where(Xiq < 1e-6, 1., Xiq)
        os_ = SL // 3; oe_ = min(os_ + 100, SL)
        r   = predict_sequence(model, xns, yrs, os_, oe_, dv_iqr, dv_median, DEVICE)
        norm_d.append(r['drift_m'])
        norm_preds.append(r['dv_pred_ms'][os_:oe_])
        norm_trues.append(r['dv_true_ms'][os_:oe_])
        vr = torch.from_numpy(yrs).float(); vp_ = torch.zeros(SL, 3); last = vr[0].clone()
        for t in range(SL):
            if t >= os_ and t < oe_: vp_[t] = last
            else:
                if t < os_: vp_[t] = vr[t]; last = vr[t].clone()
        norm_vprevs.append(vp_[os_:oe_].numpy())

# T1: v_prev ablation
print('\nT1 — v_prev Ablation (R20:1.00x, target>1.5x)')
abl_d = []
with torch.no_grad():
    for res in all_res[:10]:
        si = res['seq_idx']; st = int(vi2[si])
        xrs = Xv2[st:st+SL]; yrs = Yv2[st:st+SL]
        xns = (xrs - Xmed) / np.where(Xiq < 1e-6, 1., Xiq)
        xns2 = xns.copy(); xns2[:, :, 10:13] = 0.
        r_a = predict_sequence(model, xns2, yrs, SL//3, min(SL//3+100, SL), dv_iqr, dv_median, DEVICE)
        abl_d.append(r_a['drift_m'])
r1 = np.mean(abl_d) / (np.mean(norm_d) + 1e-6)
print(f'  Normal:{np.mean(norm_d):.2f}m  Ablated:{np.mean(abl_d):.2f}m  Ratio:{r1:.2f}x')
print(f'  {"✓ v_prev pathway used" if r1>1.5 else "~ Marginal" if r1>1.1 else "✗ Still coasting (expected — consistent with R20)"}')

# T2: IMU shuffle
print('\nT2 — IMU Shuffle (R20:1.04x)')
shuf_d = []
with torch.no_grad():
    for res in all_res[:10]:
        si = res['seq_idx']; st = int(vi2[si])
        xrs = Xv2[st:st+SL]; yrs = Yv2[st:st+SL]
        xns = (xrs - Xmed) / np.where(Xiq < 1e-6, 1., Xiq)
        os_ = SL // 3; oe_ = min(os_ + 100, SL)
        xs  = xns.copy(); perm = np.random.permutation(np.arange(os_, oe_)); xs[os_:oe_] = xs[perm]
        r_s = predict_sequence(model, xs, yrs, os_, oe_, dv_iqr, dv_median, DEVICE)
        shuf_d.append(r_s['drift_m'])
r2 = np.mean(shuf_d) / (np.mean(norm_d) + 1e-6)
print(f'  Normal:{np.mean(norm_d):.2f}m  Shuffled:{np.mean(shuf_d):.2f}m  Ratio:{r2:.2f}x')

# T3: Per-axis variance & bias
print('\nT3 — Per-Axis Variance & Bias (R20:vy_y=3.55x, target [0.5,3.0])')
pa = np.concatenate(norm_preds); ta = np.concatenate(norm_trues); pv_a = np.concatenate(norm_vprevs)
print(f"  {'Axis':<8} {'Bias':>9} {'Ratio':>7} {'|pred-vp|':>10}  Status")
print('  ' + '-' * 45)
for i, ax_n in enumerate(['vy_x', 'vy_y', 'vy_z']):
    bias  = np.mean(pa[:, i] - ta[:, i])
    ts    = np.std(ta[:, i]); ps = np.std(pa[:, i]); ratio = ps / (ts + 1e-8)
    vpe   = np.abs(pa[:, i] - pv_a[:, i]).mean()
    ok    = 0.5 <= ratio <= 3.0
    print(f'  {ax_n:<8} {bias:>+9.4f} {ratio:>7.2f} {vpe:>10.4f}  {"✓" if ok else "✗"}')

# T4: Turn vs Straight RMSE
print('\nT4 — Turn/Straight RMSE')
turn_e2, str_e2 = [], []
for i, res in enumerate(all_res[:10]):
    si = res['seq_idx']; st = int(vi2[si])
    xrs = Xv2[st:st+SL]; os_ = SL // 3; oe_ = min(os_ + 100, SL)
    gyro = np.linalg.norm(xrs[os_:oe_, WIN_LEN//2, 3:6], axis=-1)
    p = norm_preds[i]; t_ = norm_trues[i]
    for j in range(min(len(gyro), len(p))):
        e2 = np.mean((p[j] - t_[j]) ** 2)
        if gyro[j] > TURN_GYRO_THR: turn_e2.append(e2)
        elif gyro[j] >= ZUPT_GYRO_THR: str_e2.append(e2)
if turn_e2 and str_e2:
    tr = np.sqrt(np.mean(turn_e2)); sr = np.sqrt(np.mean(str_e2))
    print(f'  Turn:{tr:.4f}  Straight:{sr:.4f}  Ratio:{tr/(sr+1e-8):.3f}x  ({len(turn_e2)} turn / {len(str_e2)} str windows)')

# T5: LSTM ablation (analogous to TCN ablation in MARSNet)
print('\nT5 — LSTM Ablation (R20 TCN:92.36x — key comparison)')
lstm_d = []; orig = {k: v.clone() for k, v in model.state_dict().items()}
with torch.no_grad():
    for name, param in model.named_parameters():
        if 'lstm' in name: param.data.zero_()   # zero all LSTM weight/bias tensors
    for res in all_res[:10]:
        si = res['seq_idx']; st = int(vi2[si])
        xrs = Xv2[st:st+SL]; yrs = Yv2[st:st+SL]
        xns = (xrs - Xmed) / np.where(Xiq < 1e-6, 1., Xiq)
        r_t = predict_sequence(model, xns, yrs, SL//3, min(SL//3+100, SL), dv_iqr, dv_median, DEVICE)
        lstm_d.append(r_t['drift_m'])
model.load_state_dict(orig)
r5 = np.mean(lstm_d) / (np.mean(norm_d) + 1e-6)
print(f'  Normal:{np.mean(norm_d):.2f}m  LSTM-ablated:{np.mean(lstm_d):.2f}m  Ratio:{r5:.2f}x')
print(f'  {"✓ LSTM contributing" if r5>5 else "~ Minor" if r5>1.5 else "✗ LSTM not contributing"}')
print(f'  Interpretation: if ratio >> 5x, temporal task generalises to LSTM backbone.')
print(f'  If ratio << 92x (R20 TCN), TCN+ALiBi has architectural advantage.')

# T6: v_prev tracking
print('\nT6 — v_prev Tracking (R20:0.258, target<0.05)')
se_list, te_list = [], []
with torch.no_grad():
    for i, res in enumerate(all_res[:20]):
        si = res['seq_idx']; st = int(vi2[si])
        xrs = Xv2[st:st+SL]; yrs = Yv2[st:st+SL]
        xns = (xrs - Xmed) / np.where(Xiq < 1e-6, 1., Xiq)
        os_ = SL // 3; oe_ = min(os_ + 100, SL)
        r   = predict_sequence(model, xns, yrs, os_, oe_, dv_iqr, dv_median, DEVICE)
        gyro    = np.linalg.norm(xrs[os_:oe_, WIN_LEN//2, 3:6], axis=-1)
        pred_y  = r['dv_pred_ms'][os_ - os_:oe_ - os_, 1]
        idx     = min(i, len(norm_vprevs) - 1); vprev_y = norm_vprevs[idx][:, 1]
        err_y   = np.abs(pred_y - vprev_y[:len(pred_y)])
        grp     = res.get('group', '?')
        if 'straight' in grp or grp == 'FALSE-ALARM': se_list.extend(err_y.tolist())
        elif grp == 'TURN': te_list.extend(err_y.tolist())
se = np.mean(se_list) if se_list else float('nan')
te = np.mean(te_list) if te_list else float('nan')
print(f'  Straight |v_pred_y−v_prev_y|: {se:.4f} m/s  (target<0.05, R20=0.258)')
print(f'  Turn     |v_pred_y−v_prev_y|: {te:.4f} m/s  (want higher than straight)')

print('\n' + '=' * 65)
print(f'  SUMMARY vs R20 (MARSNet)')
print(f'  {"Metric":<28} {"R20":>7} {"LSTM":>7}')
print(f'  {"-"*44}')
print(f'  {"Mean drift":<28} {"6.56m":>7} {f"{ma.mean():.2f}m":>7}')
print(f'  {"% under 5m":<28} {"79.7%":>7} {f"{100*np.mean(ma<5):.1f}%":>7}')
print(f'  {"TCN/LSTM ablation":<28} {"92.36x":>7} {f"{r5:.2f}x":>7}')
print(f'  {"v_prev tracking":<28} {"0.258":>7} {f"{se:.4f}":>7}')
print('=' * 65)

## For Paper figures

In [ ]:
import pandas as pd
rows = [{'seq_idx': r['seq_idx'], 'drift_m': r['drift_m'],
         'naive_drift_m': r['naive_drift_m'], 'group': r['group'],
         'outage_start': r['outage_start']} for r in all_res]
pd.DataFrame(rows).to_csv(os.path.join(PLOT_DIR, 'val_seq_results.csv'), index=False)
print(f'Saved val_seq_results.csv ({len(rows)} rows)')

In [ ]:
# 2. Save path arrays for the three paper example sequences
for r in all_res:
    if r['seq_idx'] in [41, 47, 53]:
        np.save(os.path.join(PLOT_DIR, f"path_seq{r['seq_idx']}.npy"), {
            'pos_pred': r['pos_pred'][:, :2], 'pos_true': r['pos_gt'][:, :2],
            'outage_start': r['outage_start'], 'drift_m': r['drift_m'],
            'naive_drift_m': r['naive_drift_m']})